# 09 — PaddleOCR-VL-1.6 Fine-tuned Checkpoint Benchmark — Google Colab Pro

Evaluates the exported **epoch1** and **epoch2** full-SFT snapshots on validation CER, selects the better checkpoint, then optionally runs the frozen test once.

If epoch2 CER is lower than epoch1 CER, the result is evidence that an optional third epoch may be worth testing. This notebook does **not** automatically train epoch3.

## 0. Install inference runtime

In [ ]:
%pip install -q "paddlepaddle-gpu==3.2.1" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%pip install -q -U "paddleocr[doc-parser]>=3.6.0" "kagglehub>=1.0.2" "jiwer>=4.0.0" nvidia-ml-py
%pip install -q https://paddle-whl.bj.bcebos.com/nightly/cu126/safetensors/safetensors-0.6.2.dev0-cp38-abi3-linux_x86_64.whl
%pip install -q --force-reinstall opencv-python-headless "numpy==1.26.4"

## 1. Data/evaluator

In [ ]:
import os, json, time, random, platform, unicodedata, gc, math, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from jiwer import cer, wer

from google.colab import drive
drive.mount('/content/drive')

# Optional Colab secret. KaggleHub can also prompt/authenticate through its normal flow.
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except Exception:
    pass

import kagglehub

SEED = 42
RAW_HANDLE = 'ntklinhfitus/uit-hwdb'
MANIFEST_HANDLE = 'ntklinhfitus/uit-hwdb-manifest'
PROJECT_ROOT = Path('/content/drive/MyDrive/vlm_handwriting_ocr')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Try a partial raw download first. Fall back to the full Kaggle dataset if the
# installed KaggleHub/runtime does not accept directory-level download.
try:
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE, path='UIT_HWDB_line'))
except Exception as e:
    print('Partial raw download unavailable, falling back to full dataset:', repr(e))
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE))
manifest_download = Path(kagglehub.dataset_download(MANIFEST_HANDLE))

def locate_raw_line_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    candidates=[p for p in pool if (p/'train_data').is_dir() and (p/'test_data').is_dir()]
    assert candidates, f'Cannot locate UIT-HWDB-line train_data/test_data under {base}'
    candidates.sort(key=lambda p: ('UIT_HWDB_line' not in str(p), len(str(p))))
    return candidates[0]

def locate_manifest_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    for p in pool:
        if all((p/f).exists() for f in ['train.csv','val.csv','test.csv']):
            return p
    raise FileNotFoundError(f'Cannot locate train.csv/val.csv/test.csv under {base}')

RAW_ROOT=locate_raw_line_root(raw_download)
MANIFEST_ROOT=locate_manifest_root(manifest_download)
print('RAW_ROOT      =',RAW_ROOT)
print('MANIFEST_ROOT =',MANIFEST_ROOT)

In [ ]:
train_df=pd.read_csv(MANIFEST_ROOT/'train.csv')
val_df=pd.read_csv(MANIFEST_ROOT/'val.csv')
test_df=pd.read_csv(MANIFEST_ROOT/'test.csv')
EXPECTED={'train':6346,'validation':682,'test':201}
assert len(train_df)==EXPECTED['train'],len(train_df)
assert len(val_df)==EXPECTED['validation'],len(val_df)
assert len(test_df)==EXPECTED['test'],len(test_df)
required={'writer_id','filename','relative_path','text'}
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=required-set(frame.columns)
    assert not missing,f'{name} missing columns: {missing}'
train_writers=set(train_df.writer_id); val_writers=set(val_df.writer_id); test_writers=set(test_df.writer_id)
assert train_writers.isdisjoint(val_writers)
assert train_writers.isdisjoint(test_writers)
assert val_writers.isdisjoint(test_writers)

def resolve_image_path(row):
    return RAW_ROOT/str(row['relative_path'])
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=[str(resolve_image_path(r)) for _,r in frame.iterrows() if not resolve_image_path(r).exists()]
    assert not missing,f'{name}: missing image paths, e.g. {missing[:3]}'
print(f'Train      : {len(train_df)} samples | {len(train_writers)} writers')
print(f'Validation : {len(val_df)} samples | {len(val_writers)} writers')
print(f'Test       : {len(test_df)} samples | {len(test_writers)} writers')
print('✅ Frozen writer-disjoint split verified.')

In [ ]:
def normalize_for_eval(text):
    # Strict OCR evaluation: Unicode NFC only.
    return unicodedata.normalize('NFC',str(text))

def compute_metrics(gt_list,pred_list):
    if len(gt_list)!=len(pred_list) or len(gt_list)==0:
        raise ValueError('GT/prediction lists must have the same non-zero length.')
    gt=[normalize_for_eval(x) for x in gt_list]
    pred=[normalize_for_eval(x) for x in pred_list]
    exact=sum(g==p for g,p in zip(gt,pred))/len(gt)
    return {'CER':float(cer(gt,pred)),'WER':float(wer(gt,pred)),'Exact_Line_Accuracy':float(exact),'N':int(len(gt))}
assert compute_metrics(['Việt Nam'],['Việt Nam'])['CER']==0.0
assert compute_metrics(['Biển Đông.'],['Biển đông.'])['CER']>0.0
SMOKE_SIZE=20
smoke_df=val_df.sample(n=SMOKE_SIZE,random_state=SEED).sort_index().reset_index(drop=True)
print('✅ Strict evaluator + fixed 20-sample validation smoke set ready.')

## 2. Find epoch snapshots

In [ ]:
import paddle,threading
from paddleocr import PaddleOCRVL
from pynvml import nvmlInit,nvmlDeviceGetHandleByIndex,nvmlDeviceGetMemoryInfo
nvmlInit(); _h=nvmlDeviceGetHandleByIndex(0)
CKPT_ROOT=PROJECT_ROOT/'checkpoints'/'paddleocr_vl_1_6'; RESULT_DIR=PROJECT_ROOT/'results'/'paddleocr_vl_1_6'/'finetuned'; RESULT_DIR.mkdir(parents=True,exist_ok=True)
checkpoints=[p for p in [CKPT_ROOT/'epoch1',CKPT_ROOT/'epoch2'] if p.is_dir()]
assert len(checkpoints)>=2,f'Need epoch1 and epoch2 snapshots in {CKPT_ROOT}. Found: {checkpoints}'
for p in checkpoints: print(' -',p)

## 3. Pipeline/evaluation helpers

In [ ]:
def build_pipeline(model_dir):
    return PaddleOCRVL(pipeline_version='v1.6',vl_rec_model_name='PaddleOCR-VL-1.6-0.9B',vl_rec_model_dir=str(model_dir),use_layout_detection=False,use_doc_orientation_classify=False,use_doc_unwarping=False,device='gpu:0')

def extract_text(obj):
    payload=obj.json
    if callable(payload): payload=payload()
    if isinstance(payload,dict) and isinstance(payload.get('res'),dict): payload=payload['res']
    blocks=payload.get('parsing_res_list',[]) if isinstance(payload,dict) else []
    texts=[str(b.get('block_content','')) for b in blocks if str(b.get('block_content','')).strip()]
    if texts: return '\n'.join(texts).strip()
    raise RuntimeError(f'Cannot extract parsing_res_list from {payload.keys() if isinstance(payload,dict) else type(payload)}')

def predict_one(pipe,image_path):
    vals=[]; stop=threading.Event()
    def monitor():
        while not stop.is_set(): vals.append(nvmlDeviceGetMemoryInfo(_h).used); time.sleep(.03)
    t=threading.Thread(target=monitor,daemon=True); t.start(); t0=time.perf_counter()
    try: results=list(pipe.predict(str(image_path),use_layout_detection=False,prompt_label='ocr',use_queues=False,max_new_tokens=256))
    finally: lat=time.perf_counter()-t0; stop.set(); t.join(timeout=1)
    assert len(results)==1; return extract_text(results[0]),lat,(max(vals)/1024**3 if vals else None)

def eval_frame(pipe,frame,split_name):
    rows=[]; peaks=[]; started=time.perf_counter()
    for i,(_,row) in enumerate(frame.iterrows(),1):
        pred,lat,peak=predict_one(pipe,resolve_image_path(row)); gt=normalize_for_eval(row['text']); pred=normalize_for_eval(pred); peaks.append(peak or 0); rows.append({'writer_id':int(row['writer_id']),'filename':row['filename'],'ground_truth':gt,'prediction':pred,'sample_CER':float(cer(gt,pred)),'latency_sec':float(lat)})
        if i%50==0 or i==len(frame): print(f'[{i}/{len(frame)}]')
    out=pd.DataFrame(rows); m=compute_metrics(out.ground_truth.tolist(),out.prediction.tolist()); m.update({'split':split_name,'latency_mean_sec':float(out.latency_sec.mean()),'total_runtime_sec':float(time.perf_counter()-started),'peak_gpu_memory_used_gb_nvml':float(max(peaks)) if peaks else None}); return out,m

## 4. Smoke both checkpoints

In [ ]:
for ckpt in checkpoints:
    print('\nSMOKE',ckpt.name); pipe=build_pipeline(ckpt); _,m=eval_frame(pipe,smoke_df,'validation_smoke20'); print(m); del pipe; gc.collect()

## 5. Full validation and checkpoint selection

In [ ]:
RUN_CHECKPOINT_SELECTION=True; summary=[]
if RUN_CHECKPOINT_SELECTION:
    for ckpt in checkpoints:
        print('\nVALIDATION',ckpt.name); pipe=build_pipeline(ckpt); preds,m=eval_frame(pipe,val_df.reset_index(drop=True),'validation'); m['checkpoint']=str(ckpt); summary.append(m); preds.to_csv(RESULT_DIR/f'{ckpt.name}_val_predictions.csv',index=False); (RESULT_DIR/f'{ckpt.name}_val_metrics.json').write_text(json.dumps(m,ensure_ascii=False,indent=2),encoding='utf-8'); del pipe; gc.collect()
    summary_df=pd.DataFrame(summary).sort_values('CER').reset_index(drop=True); display(summary_df); BEST_CHECKPOINT=Path(summary_df.iloc[0].checkpoint); (RESULT_DIR/'best_checkpoint.json').write_text(json.dumps({'best_checkpoint':str(BEST_CHECKPOINT),'selection_metric':'validation CER','validation_CER':float(summary_df.iloc[0].CER)},indent=2),encoding='utf-8')
    epoch1=float(summary_df.loc[summary_df.checkpoint.str.endswith('epoch1'),'CER'].iloc[0]); epoch2=float(summary_df.loc[summary_df.checkpoint.str.endswith('epoch2'),'CER'].iloc[0]); print('epoch1 CER=',epoch1,'epoch2 CER=',epoch2); print('Optional epoch3 decision:', 'CONSIDER epoch3' if epoch2<epoch1 else 'STOP at <=2 epochs')

## 6. Frozen test with best checkpoint

In [ ]:
RUN_TEST_BENCHMARK=False
if RUN_TEST_BENCHMARK:
    if 'BEST_CHECKPOINT' not in globals(): BEST_CHECKPOINT=Path(json.loads((RESULT_DIR/'best_checkpoint.json').read_text())['best_checkpoint'])
    pipe=build_pipeline(BEST_CHECKPOINT); preds,m=eval_frame(pipe,test_df.reset_index(drop=True),'test'); m['checkpoint']=str(BEST_CHECKPOINT); preds.to_csv(RESULT_DIR/'best_test_predictions.csv',index=False); (RESULT_DIR/'best_test_metrics.json').write_text(json.dumps(m,ensure_ascii=False,indent=2),encoding='utf-8'); print(json.dumps(m,ensure_ascii=False,indent=2))
else: print('Frozen test disabled.')